# ⬡ Dabba DevOps — Fine-tune Llama 3.2 3B into a DevOps-only assistant

**Before running: Runtime → Change runtime type → T4 GPU → Save**

A **small, focused** model (3B params) trained specifically on:
- Kubernetes & Infrastructure as Code (Terraform, Ansible, Helm)
- CI/CD pipelines (GitHub Actions, GitLab CI, Jenkins, ArgoCD)
- Shell scripting & Linux administration
- Cloud platforms (AWS, GCP, Azure)
- Monitoring & observability (Prometheus, Grafana, alerting)

Being narrow instead of general-purpose means it should punch above its
weight *in this domain specifically*, and trains much faster than the 8B
general Dabba model.

**Time on free T4:** ~1-1.5 hours (3B is much smaller than 8B)

**⚠️ Always use Save & Run All / keep the tab active** — see the resume
instructions at the bottom if you get disconnected.

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())
# Should show: Tesla T4, 15360 MiB
# If it says CPU only / errors → Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Cell 2: Install (run once, ~3 min) ────────────────────────────────────────
%%capture
!pip install unsloth
!pip install --no-deps "trl>=0.9.0" peft accelerate bitsandbytes datasets
print('done')

In [ ]:
# ── Cell 3: Verify install ─────────────────────────────────────────────────────
import unsloth, trl, peft, transformers
print(f'unsloth {unsloth.__version__}  trl {trl.__version__}  peft {peft.__version__}  transformers {transformers.__version__}')
print('✓ All packages ready')

In [ ]:
# ── Cell 4: Load Llama 3.2 3B in 4-bit ────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,      # auto-detect
    load_in_4bit   = True,
)

used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✓ Model loaded — VRAM used: {used:.1f}GB / {total:.1f}GB')

In [ ]:
# ── Cell 5: Add LoRA adapters ──────────────────────────────────────────────────
# 3B is much smaller than 8B, so we can afford a higher rank (r=32) for the
# same T4 budget — helps it absorb the specialized DevOps vocabulary better.
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 32,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f'✓ LoRA ready — trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)')

## Cell 6 — Build the DevOps dataset

This is the important part. Rather than trusting an unverified HuggingFace
dataset name (which could 404 and waste your session), the DevOps-specific
content below is **hand-written directly in this notebook** — guaranteed to
exist and cover exactly the five areas you asked for. It's repeated a few
times in the final mix so it carries real weight against the general-purpose
data, instead of being diluted into irrelevance.

**To add your own runbooks/docs:** see the optional upload cell right after —
drop a `.jsonl` file with `{"instruction": ..., "output": ...}` per line and
it merges in automatically.

In [ ]:
# ── Cell 6: Hand-written DevOps instruction set ───────────────────────────────
# Edit/extend this list freely — each entry is one training example.
DEVOPS_EXAMPLES = [
    # ── Kubernetes & IaC ──
    {"instruction": "Write a Kubernetes Deployment YAML for an nginx app with 3 replicas, exposing port 80.",
     "output": "apiVersion: apps/v1\nkind: Deployment\nmetadata:\n  name: nginx-deployment\n  labels:\n    app: nginx\nspec:\n  replicas: 3\n  selector:\n    matchLabels:\n      app: nginx\n  template:\n    metadata:\n      labels:\n        app: nginx\n    spec:\n      containers:\n      - name: nginx\n        image: nginx:latest\n        ports:\n        - containerPort: 80"},
    {"instruction": "A pod is stuck in CrashLoopBackOff. How do I debug it?",
     "output": "1. Check pod status: `kubectl get pod <name> -o wide`\n2. View logs: `kubectl logs <name> --previous` (shows the last crashed container's logs)\n3. Describe the pod for events: `kubectl describe pod <name>` — look at the Events section for OOMKilled, failed liveness probes, or image pull errors\n4. If it's a config issue, exec into a debug pod with the same image: `kubectl run debug --image=<image> -it --rm -- sh`\n5. Check resource limits — CrashLoopBackOff often means the container OOM'd; raise memory limits or fix a memory leak\n6. Verify the container's entrypoint/command actually starts a long-running process, not one that exits immediately"},
    {"instruction": "Write a Terraform resource block to create an AWS S3 bucket with versioning enabled.",
     "output": "resource \"aws_s3_bucket\" \"data\" {\n  bucket = \"my-app-data-bucket\"\n}\n\nresource \"aws_s3_bucket_versioning\" \"data_versioning\" {\n  bucket = aws_s3_bucket.data.id\n  versioning_configuration {\n    status = \"Enabled\"\n  }\n}"},
    {"instruction": "Explain the difference between a Kubernetes Deployment and a StatefulSet.",
     "output": "Deployments manage stateless, interchangeable pods — any replica can be replaced by another with no identity requirement, pods get random names, and there's no guaranteed ordering for scaling/updates. StatefulSets are for stateful workloads (databases, message queues) — each pod gets a stable, unique network identity (pod-0, pod-1, ...) and stable persistent storage tied to that identity that survives rescheduling. StatefulSets also scale/update pods in strict order (0, 1, 2, ...) instead of in parallel. Use Deployment for web servers/APIs; use StatefulSet for databases like Postgres, Kafka, or Elasticsearch."},
    {"instruction": "Write an Ansible playbook task to install and start nginx on Debian-based hosts.",
     "output": "- name: Install and start nginx\n  hosts: webservers\n  become: true\n  tasks:\n    - name: Install nginx\n      apt:\n        name: nginx\n        state: present\n        update_cache: true\n\n    - name: Ensure nginx is running and enabled\n      service:\n        name: nginx\n        state: started\n        enabled: true"},
    {"instruction": "How do I roll back a Kubernetes deployment to the previous version?",
     "output": "`kubectl rollout undo deployment/<name>` rolls back to the immediately previous revision. To see revision history first: `kubectl rollout history deployment/<name>`. To roll back to a specific revision: `kubectl rollout undo deployment/<name> --to-revision=<N>`. Watch the rollback progress with `kubectl rollout status deployment/<name>`."},

    # ── CI/CD ──
    {"instruction": "Write a GitHub Actions workflow that runs pytest on every push to main.",
     "output": "name: Tests\n\non:\n  push:\n    branches: [main]\n\njobs:\n  test:\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: '3.11'\n      - name: Install dependencies\n        run: pip install -r requirements.txt\n      - name: Run tests\n        run: pytest --tb=short -q"},
    {"instruction": "Write a .gitlab-ci.yml with build, test, and deploy stages.",
     "output": "stages:\n  - build\n  - test\n  - deploy\n\nbuild:\n  stage: build\n  script:\n    - docker build -t myapp:$CI_COMMIT_SHA .\n\ntest:\n  stage: test\n  script:\n    - pytest --tb=short\n\ndeploy:\n  stage: deploy\n  script:\n    - kubectl set image deployment/myapp myapp=myapp:$CI_COMMIT_SHA\n  only:\n    - main"},
    {"instruction": "Explain what ArgoCD does and how GitOps differs from a traditional CI/CD push-based pipeline.",
     "output": "ArgoCD is a GitOps continuous-delivery tool for Kubernetes: it continuously watches a Git repo containing your desired cluster state (manifests/Helm charts) and automatically syncs the live cluster to match it. In traditional push-based CI/CD, the pipeline actively pushes changes to the cluster (e.g. `kubectl apply` from a Jenkins job) — the pipeline has direct write access to production. In GitOps (pull-based), the cluster itself pulls changes via an agent (ArgoCD) watching Git; Git becomes the single source of truth, changes are auditable via commit history, and rollback is just reverting a Git commit. This is more secure (no external system needs cluster credentials) and self-healing (ArgoCD detects and corrects drift automatically)."},
    {"instruction": "Write a Jenkinsfile (declarative) with build and deploy stages for a Docker app.",
     "output": "pipeline {\n    agent any\n    stages {\n        stage('Build') {\n            steps {\n                sh 'docker build -t myapp:${BUILD_NUMBER} .'\n            }\n        }\n        stage('Test') {\n            steps {\n                sh 'docker run myapp:${BUILD_NUMBER} pytest'\n            }\n        }\n        stage('Deploy') {\n            steps {\n                sh 'kubectl set image deployment/myapp myapp=myapp:${BUILD_NUMBER}'\n            }\n        }\n    }\n}"},

    # ── Shell & Linux ──
    {"instruction": "Write a bash script that finds and deletes log files older than 7 days in /var/log/myapp.",
     "output": "#!/bin/bash\nset -euo pipefail\n\nLOG_DIR=\"/var/log/myapp\"\nDAYS=7\n\nfind \"$LOG_DIR\" -type f -name '*.log' -mtime +\"$DAYS\" -print -delete\n\necho \"Deleted logs older than $DAYS days from $LOG_DIR\""},
    {"instruction": "How do I find which process is using the most memory on a Linux server?",
     "output": "Quick one-liner: `ps aux --sort=-%mem | head -10` shows the top 10 processes by memory %. For more detail: `top` (interactive, sorted by CPU by default — press `M` to sort by memory) or `htop` if installed. To see memory in absolute terms: `ps aux --sort=-rss | head -10` (RSS = resident set size in KB). For a system-wide breakdown: `free -h` shows total/used/free/cache, and `smem` (if installed) gives more accurate per-process memory accounting than `ps` since it accounts for shared memory correctly."},
    {"instruction": "Write a systemd unit file for a Python app that should restart automatically if it crashes.",
     "output": "[Unit]\nDescription=My Python App\nAfter=network.target\n\n[Service]\nType=simple\nUser=appuser\nWorkingDirectory=/opt/myapp\nExecStart=/usr/bin/python3 /opt/myapp/main.py\nRestart=on-failure\nRestartSec=5\n\n[Install]\nWantedBy=multi-user.target"},
    {"instruction": "Write a cron entry that runs a backup script every day at 2:30 AM.",
     "output": "30 2 * * * /opt/scripts/backup.sh >> /var/log/backup.log 2>&1\n\nExplanation: minute=30, hour=2, every day-of-month, every month, every day-of-week. The output redirection appends both stdout and stderr to a log file so cron failures aren't silently lost."},
    {"instruction": "How do I check which port a process is listening on in Linux?",
     "output": "`sudo ss -tlnp | grep <port>` (modern, replaces netstat) shows TCP listening sockets with the owning process. Example: `sudo ss -tlnp | grep :8080`. Alternative: `sudo lsof -i :8080` shows exactly which process holds that port. Older systems: `sudo netstat -tlnp | grep 8080`."},

    # ── Cloud (AWS/GCP/Azure) ──
    {"instruction": "Write an AWS CLI command to list all EC2 instances that are currently running.",
     "output": "aws ec2 describe-instances \\\n  --filters \"Name=instance-state-name,Values=running\" \\\n  --query 'Reservations[*].Instances[*].[InstanceId,InstanceType,PublicIpAddress]' \\\n  --output table"},
    {"instruction": "Explain the difference between an AWS IAM Role and an IAM User.",
     "output": "An IAM User represents a permanent identity (a person or service) with long-term credentials (access key + secret, or console password) directly attached. An IAM Role has no permanent credentials — instead, it's assumed temporarily by a trusted entity (an EC2 instance, a Lambda function, another AWS account, or a federated user), which receives short-lived, auto-rotating temporary credentials via STS. Best practice: use Roles for anything running inside AWS (EC2, Lambda, ECS tasks) instead of embedding long-lived User access keys in code or config — Roles eliminate the risk of leaked long-term credentials and are easier to audit/rotate."},
    {"instruction": "Write a gcloud command to deploy a container to Cloud Run.",
     "output": "gcloud run deploy myapp \\\n  --image gcr.io/my-project/myapp:latest \\\n  --platform managed \\\n  --region us-central1 \\\n  --allow-unauthenticated \\\n  --memory 512Mi \\\n  --port 8080"},
    {"instruction": "How do I find which AWS resources are costing the most money this month?",
     "output": "Use AWS Cost Explorer: in the console under Billing → Cost Explorer, group by 'Service' with a daily/monthly granularity for the current month. Via CLI: `aws ce get-cost-and-usage --time-period Start=2026-07-01,End=2026-07-31 --granularity MONTHLY --metrics BlendedCost --group-by Type=DIMENSION,Key=SERVICE`. For a deeper breakdown, enable Cost Allocation Tags on your resources (e.g. tag by team/project) so Cost Explorer can group spend by tag, not just by service — critical once you have many resources sharing a service type like EC2."},
    {"instruction": "Write an Azure CLI command to create a resource group and a storage account inside it.",
     "output": "az group create --name myResourceGroup --location eastus\n\naz storage account create \\\n  --name mystorageacct123 \\\n  --resource-group myResourceGroup \\\n  --location eastus \\\n  --sku Standard_LRS"},

    # ── Monitoring & Observability ──
    {"instruction": "Write a Prometheus alerting rule that fires if a service's error rate exceeds 5% over 5 minutes.",
     "output": "groups:\n  - name: error-rate-alerts\n    rules:\n      - alert: HighErrorRate\n        expr: |\n          (sum(rate(http_requests_total{status=~\"5..\"}[5m])) by (service)\n          /\n          sum(rate(http_requests_total[5m])) by (service)) > 0.05\n        for: 5m\n        labels:\n          severity: critical\n        annotations:\n          summary: \"High error rate on {{ $labels.service }}\"\n          description: \"Error rate is {{ $value | humanizePercentage }} for the last 5 minutes.\""},
    {"instruction": "Explain the four golden signals of monitoring used in SRE practice.",
     "output": "The four golden signals (from Google's SRE book) are: 1) Latency — time to serve a request, tracked separately for successful vs failed requests since failures can be artificially fast. 2) Traffic — demand on the system, e.g. requests/sec or concurrent sessions. 3) Errors — rate of failed requests, whether explicit (5xx) or implicit (wrong content, policy violations). 4) Saturation — how 'full' the system is relative to capacity (CPU, memory, disk I/O, queue depth) — the earliest warning sign before latency/errors actually appear. Tracking all four together gives a complete picture: traffic explains load, saturation predicts trouble, and latency/errors show current user impact."},
    {"instruction": "Write a basic Grafana dashboard JSON panel for CPU usage from a Prometheus data source.",
     "output": "{\n  \"type\": \"timeseries\",\n  \"title\": \"CPU Usage\",\n  \"datasource\": \"Prometheus\",\n  \"targets\": [\n    {\n      \"expr\": \"100 - (avg by(instance) (rate(node_cpu_seconds_total{mode=\\\"idle\\\"}[5m])) * 100)\",\n      \"legendFormat\": \"{{instance}}\"\n    }\n  ],\n  \"fieldConfig\": {\n    \"defaults\": { \"unit\": \"percent\", \"max\": 100 }\n  }\n}"},
    {"instruction": "How does Prometheus discover and scrape targets in a Kubernetes cluster?",
     "output": "Prometheus uses Kubernetes service discovery (`kubernetes_sd_configs`) to automatically find scrape targets — it queries the Kubernetes API for pods, services, endpoints, or nodes matching configured roles, and re-discovers them periodically so it adapts as pods scale up/down. Typically pods expose metrics on a `/metrics` HTTP endpoint, and are annotated (e.g. `prometheus.io/scrape: \"true\"`, `prometheus.io/port: \"9090\"`) so Prometheus's relabeling rules know which pods to scrape and on which port. In production this is usually managed by the Prometheus Operator, which uses ServiceMonitor/PodMonitor custom resources instead of hand-written scrape configs, making target discovery declarative and GitOps-friendly."},
    {"instruction": "A Grafana panel shows 'No Data' but the Prometheus target is UP. What should I check?",
     "output": "1. Check the PromQL query itself in Prometheus's own UI first (http://prometheus:9090/graph) — if it returns no data there too, the issue is the query/metric name, not Grafana\n2. Verify the time range in Grafana matches when data actually exists (a common gotcha — dashboard defaulting to 'Last 6 hours' when the metric only started recently)\n3. Check the datasource configuration in Grafana points to the correct Prometheus URL and the datasource test succeeds (Configuration → Data Sources → Test)\n4. Verify label matchers in the query match actual label values exactly (`{job=\"myapp\"}` won't match if the actual label value is different, e.g. `\"my-app\"`) — check via `label_values(job)` in Prometheus's UI\n5. Check for typos in the metric name — use Prometheus's metric browser/autocomplete to confirm the exact name"},
]

print(f'✓ {len(DEVOPS_EXAMPLES)} hand-written DevOps examples loaded')

In [ ]:
# ── Cell 6b (optional): Upload your own runbooks/docs ─────────────────────────
# Skip this cell if you don't have your own data yet — DEVOPS_EXAMPLES above is
# enough to run the pipeline end to end.
#
# Format: a .jsonl file, one JSON object per line:
#   {"instruction": "How do I restart the payments service?", "output": "Run `systemctl restart payments` on the app server, then check `journalctl -u payments -f` for startup errors..."}
import json, os

CUSTOM_DATA_FILE = "my_devops_data.jsonl"  # upload this file into Colab's file browser first

custom_examples = []
if os.path.exists(CUSTOM_DATA_FILE):
    with open(CUSTOM_DATA_FILE) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                ex = json.loads(line)
                if "instruction" in ex and "output" in ex:
                    custom_examples.append(ex)
            except json.JSONDecodeError:
                pass
    print(f'✓ Loaded {len(custom_examples)} of your own examples from {CUSTOM_DATA_FILE}')
else:
    print(f'No {CUSTOM_DATA_FILE} found — skipping (this is fine, just uses the built-in DevOps set)')

In [ ]:
# ── Cell 7: Build the final training dataset ──────────────────────────────────
from datasets import load_dataset, concatenate_datasets, Dataset

DABBA_SYSTEM = (
    "You are Dabba DevOps, a specialized AI assistant focused entirely on DevOps, "
    "infrastructure, and cloud operations engineering. You are an expert in "
    "Kubernetes, Terraform, Ansible, CI/CD pipelines, Linux/shell administration, "
    "AWS/GCP/Azure, and monitoring tools like Prometheus and Grafana. You give "
    "direct, technically precise answers with real, runnable configuration and "
    "commands — never vague advice. You are honest about what you don't know."
)

def to_chat_text(instruction, output):
    msgs = [
        {"role": "system", "content": DABBA_SYSTEM},
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

# Repeat the hand-written DevOps set + any custom data 4x so it carries real
# weight against the larger general-purpose datasets below — otherwise ~25
# examples would be a rounding error next to 10,000+ general examples.
devops_all = (DEVOPS_EXAMPLES + custom_examples) * 4
devops_texts = [to_chat_text(ex["instruction"], ex["output"]) for ex in devops_all]
devops_ds = Dataset.from_dict({"text": devops_texts})
print(f'DevOps examples (with 4x repeat): {len(devops_ds):,}')

def fmt_alpaca(ex):
    instruction = (ex.get('instruction') or '').strip()
    inp         = (ex.get('input') or '').strip()
    output      = (ex.get('output') or '').strip()
    if not instruction or not output:
        return {'text': ''}
    user = f"{instruction}\n\n{inp}" if inp else instruction
    return {'text': to_chat_text(user, output)}

print('Downloading general-purpose datasets (verified real HF datasets)...')

# General code ability — keeps it useful for scripting beyond pure DevOps trivia
code_raw = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
code_ds  = code_raw.map(fmt_alpaca, remove_columns=code_raw.column_names)

# Small general instruction-following slice — baseline conversational competence
alpaca_raw = load_dataset("tatsu-lab/alpaca", split="train[:3000]")
alpaca_ds  = alpaca_raw.map(fmt_alpaca, remove_columns=alpaca_raw.column_names)

combined = concatenate_datasets([devops_ds, code_ds, alpaca_ds])
combined = combined.filter(lambda x: len(x['text'].strip()) > 80)
combined = combined.shuffle(seed=42)

print(f'✓ Final dataset: {len(combined):,} examples')
print(f'  ({len(devops_ds):,} DevOps-weighted / {len(code_ds):,} general code / {len(alpaca_ds):,} general instructions)')
print('Sample preview:')
print(combined[0]['text'][:400])

In [ ]:
# ── Cell 8: Train (auto-resumes from the latest checkpoint if one exists) ─────
import os
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

CHECKPOINT_DIR = "dabba-devops-checkpoints"
# 3B trains much faster than 8B — 2 epochs over this smaller, focused dataset
# tends to work better than 1 epoch over a huge generic one.
NUM_EPOCHS = 2

latest_checkpoint = None
if os.path.exists(CHECKPOINT_DIR):
    checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        latest_checkpoint = os.path.join(
            CHECKPOINT_DIR, max(checkpoints, key=lambda d: int(d.split("-")[-1]))
        )

if latest_checkpoint:
    print(f'Found existing checkpoint: {latest_checkpoint} — resuming from there.')
else:
    print('No checkpoint found — starting a fresh run.')

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = combined,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    packing        = True,
    args = SFTConfig(
        per_device_train_batch_size  = 4,   # 3B fits a bigger batch than 8B did
        gradient_accumulation_steps  = 4,
        warmup_steps                 = 30,
        num_train_epochs             = NUM_EPOCHS,
        learning_rate                = 2e-4,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 10,
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "cosine",
        seed                         = 42,
        output_dir                   = CHECKPOINT_DIR,
        save_strategy                = "steps",
        save_steps                   = 50,
        save_total_limit             = 5,
        report_to                    = "none",
    ),
)

used = torch.cuda.memory_allocated() / 1e9
print(f'VRAM before training: {used:.1f}GB')
print('🚀 Training started...')

stats = trainer.train(resume_from_checkpoint=latest_checkpoint)

print(f'\n✓ Training complete!')
print(f'  Runtime : {stats.metrics["train_runtime"]/60:.1f} min')
print(f'  Final loss: {stats.metrics["train_loss"]:.4f}')

In [ ]:
# ── Cell 9: Quick test — does it actually answer DevOps questions well? ───────
FastLanguageModel.for_inference(model)

def ask(question):
    msgs = [
        {"role": "system", "content": DABBA_SYSTEM},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=400, temperature=0.7, do_sample=True)
    reply = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'You: {question}\n{reply}\n{"-"*60}')

ask("Who are you?")
ask("Write a kubectl command to get logs from all pods matching label app=api in the last 10 minutes.")
ask("My Terraform apply is stuck saying resource already exists. How do I fix that?")

In [ ]:
# ── Cell 10: Export to GGUF — WITH a real verification check ─────────────────
# Unsloth's save_pretrained_gguf builds llama.cpp internally to quantize, and
# that step can silently fail while the surrounding code still prints "done".
# This cell checks for an ACTUAL .gguf file afterward instead of trusting that.
import os

print('Exporting to GGUF... (takes ~5-10 min)')
model.save_pretrained_gguf(
    "dabba-devops-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

gguf_files = [f for f in os.listdir('dabba-devops-gguf') if f.endswith('.gguf')]
if gguf_files:
    for f in gguf_files:
        size = os.path.getsize(f'dabba-devops-gguf/{f}') / 1e9
        print(f'  ✓ {f}  ({size:.2f} GB)')
    print('✓ GGUF export genuinely succeeded')
else:
    print('⚠ NO .gguf FILE WAS PRODUCED — the quantization step silently failed.')
    print('  This is a known Unsloth/Colab issue. Files below are the raw merged model —')
    print('  download the whole dabba-devops-gguf/ folder and convert to GGUF locally instead')
    print('  (ask Claude to do this — it can run llama.cpp conversion directly on your PC).')
    for f in os.listdir('dabba-devops-gguf'):
        print(' ', f)

In [ ]:
# ── Cell 11: Save to Google Drive ─────────────────────────────────────────────
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

dest = '/content/drive/MyDrive/dabba-devops-gguf'
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree('dabba-devops-gguf', dest)

print(f'✓ Saved to Google Drive: dabba-devops-gguf/')
print('  Download it to your PC, then follow the setup below.')

## ✅ After training — run your DevOps SLM locally

### If Cell 10 produced a real .gguf file:
```bash
export DABBA_MODEL_PATH="/home/hasheem/dabba-devops-Q4_K_M.gguf"
cd "/home/hasheem/Hasheem files/Hasheem sub foders/ai"
python3 -m dabba.api.server
```
Then in the model picker (F2 in the TUI, or the model chip in VSCode), it'll
show up as your local `dabba` model once pointed at this file.

### If Cell 10 warned no .gguf was produced:
Download the whole `dabba-devops-gguf/` folder (raw merged safetensors) and
tell Claude the local folder path — the same local llama.cpp conversion
pipeline already set up for the main Dabba 8B model works for this one too.

---

### Resuming after a disconnect
Cell 8 auto-detects the latest checkpoint in `dabba-devops-checkpoints/` — no
manual checkpoint entry needed. If Colab fully resets (local disk wiped),
check Google Drive for `dabba-devops-gguf/` from a prior save, or just
re-run from Cell 1 — 3B trains fast enough that a full restart isn't too costly.